In [42]:
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
from pathlib import Path

### 1. 讀取資料(過濾review_id重複資料)+資料處理

In [43]:
connection_url = URL.create(
    drivername="mysql+pymysql",
    username="root",          
    password="a24967679",    
    host="localhost",
    port=3306,
    database="olist_staging"
)

In [44]:
engine = create_engine(connection_url)

In [45]:
sql = """
WITH review_deduplicated AS (
    SELECT
        review_id,
        review_score,
        review_comment_title,
        review_comment_message,

        ROW_NUMBER() OVER (
            PARTITION BY review_id
            ORDER BY order_id
        ) AS rn

    FROM olist_staging.stg_reviews
)

SELECT
    review_id,
    review_score,
    review_comment_title,
    review_comment_message
    
FROM review_deduplicated
WHERE rn = 1
  AND review_score BETWEEN 1 AND 5
  AND (
        NULLIF(TRIM(review_comment_title), '') IS NOT NULL
        OR
        NULLIF(TRIM(review_comment_message), '') IS NOT NULL
      )
"""

In [46]:
df_review = pd.read_sql(sql,engine)

In [47]:
df_review.head(5)

,review_id,review_score,review_comment_title,review_comment_message
0,00020c7512a52e92212f12d3e37513c0,5,Entrega rápida!,A entrega foi super rápida e o pendente é lind...
1,00046a69550325aea5fb89f65c7387f2,5,SUPER RECOMENDO,"GOSTEI DA CAPINHA DE CELULAR, VEIO COMO EU ESP..."
2,0005534973388c830bb858cfba83b17b,5,None,otimo produto. prazo cumprido. sabor tambem mu...
3,00055e36e9608fe969231e551983a69c,5,None,O produto foi entregue muito antes do esperado...
4,0005949d4c047d64863a6874338139ba,5,None,"Bom eu já sabia que era,mas é muito bonito.rec..."


In [48]:
print("資料筆數：", len(df_review))
print("review_id 重複數：", df_review["review_id"].duplicated().sum())

資料筆數： 42380
review_id 重複數： 0


In [49]:
# 合併title + message

df_review["review_text_pt"] = (
    df_review["review_comment_title"].fillna("").str.strip()
    + " "
    + df_review["review_comment_message"].fillna("").str.strip()
).str.strip()

In [50]:
df_review.isna().sum()

review_id                     0
review_score                  0
review_comment_title      30862
review_comment_message     1720
review_text_pt                0
dtype: int64

### 2. 資料抽樣

In [51]:
print("抽樣前各星等數量：\n")

summary = (
    pd.DataFrame({
        "count": df_review["review_score"].value_counts().sort_index(),
        "percent": (
            df_review["review_score"]
            .value_counts(normalize=True)
            .sort_index()
            .mul(100)
            .round(2)
        )
    })
)

summary["percent"] = summary["percent"].astype(str) + "%"

print(summary)

抽樣前各星等數量：

              count percent
review_score               
1              8723  20.58%
2              2138   5.04%
3              3615   8.53%
4              6238  14.72%
5             21666  51.12%


In [52]:
# 進行星等分層抽樣
sample_sizes = {
    1: 200,
    2: 150,
    3: 150,
    4: 100,
    5: 100
}

sample_list = []

for score, target_n in sample_sizes.items():

    score_data = df_review[
        df_review["review_score"] == score
    ]


    score_sample = score_data.sample(
        n=target_n,
        replace=False,
        random_state=42
    )

    sample_list.append(score_sample)

review_sample = pd.concat(
    sample_list,
    ignore_index=True
)

In [53]:
review_sample["review_score"].value_counts()

review_score
1    200
2    150
3    150
4    100
5    100
Name: count, dtype: int64

In [54]:
review_sample

,review_id,review_score,review_comment_title,review_comment_message,review_text_pt
0,13164d9da17abb1a513eb773bff4f274,1,Recebi pedido incompleto,Paguei por 2 unidades do Perfume Miss Gabriela...,Recebi pedido incompleto Paguei por 2 unidades...
1,b190c1c98294a0aa6beca27daff6886c,1,Produto mal embalado,Produto chegou com risco e manchado.,Produto mal embalado Produto chegou com risco ...
2,ecf05388b2201007af96097c09e68c6d,1,None,recebi o produto quebrado e aguardo retorno da...,recebi o produto quebrado e aguardo retorno da...
3,47a4f48be34dd4a1c24b24a173ed1f6d,1,None,Minha compra foi entregue para outra pessoa.\n...,Minha compra foi entregue para outra pessoa.\n...
4,f87e60d60e2e4f3e9e761a395a73fc56,1,None,Faltou item descrito\nMe ligue 34 99978 4951,Faltou item descrito\nMe ligue 34 99978 4951
...,...,...,...,...,...
695,5d60a8a8898df2ee077a134f93d4b3e7,5,Muito rapido,Excelente!!!,Muito rapido Excelente!!!
696,943a97cf91be1385859cfb55b77428b4,5,None,Muito boas. Obrigado.,Muito boas. Obrigado.
697,b326f67288f003f81c50333c7647f723,5,.,.,. .
698,75674bd25968d7b9303b48d9eb26b9f0,5,Ótimo,None,Ótimo


### 3. 存檔

In [55]:


output_dir = Path(
    "/Users/kaiping/Desktop/olist_project/"
    "Data_Understanding/review_analysis"
)


output_path = output_dir / "review_manual_annotation_v1.csv"

output_columns = [
    "review_id",
    "review_score",
    "review_text_pt"
]

review_sample[output_columns].to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"  # 避免 Excel 開啟繁體中文時亂碼
)
if output_path.exists():
    print("檔案已存在，本次不重新輸出：", output_path)
else:
    review_sample[output_columns].to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig"
    )

    print("輸出成功：", output_path)
    print("輸出筆數：", len(review_sample))

檔案已存在，本次不重新輸出： /Users/kaiping/Desktop/olist_project/Data_Understanding/review_analysis/review_manual_annotation_v1.csv
